# 텍스트마이닝 ex01 · 한국어 혐오표현 감성분석

- **주제** : 한국어 문장이 *혐오 표현*인지 *정상 문장*인지 분류하는 모델 만들기
- **데이터** : Smilegate AI **UnSmile** 데이터셋 (`data/unsmile_train_v1.0.tsv`, `data/unsmile_valid_v1.0.tsv`)
- **파이프라인** : 데이터 수집 → 정제 → 토큰화 → 특징값 추출(TF-IDF) → 모델링(Logistic Regression) → 평가

| 구분 | 건수 | 혐오(clean=0) | 정상(clean=1) |
|---|---|---|---|
| train | 15,005 | 11,266 | 3,739 |
| test | 3,737 | 2,802 | 935 |

### 목차
0. 환경 준비
1. 데이터 수집 (불러오기)
2. 데이터 정제 (정규표현식 · 띄어쓰기 교정 · 이모지 제거)
3. 토큰화 (형태소 분석 + 품사 태깅)
4. 특징값 추출 (TF-IDF)
5. 모델링 & 평가
6. 오늘의 결과 정리

---
## 0. 환경 준비

- `konlpy` : 한국어 형태소 분석기 모음 (Java 기반 → **JDK 설치 + JAVA_HOME 설정** 필요)
- `kiwipiepy` : 띄어쓰기 교정
- `emoji` : 이모지 제거
- GPU는 이번 실습(로지스틱 회귀)에 필수는 아니고 환경 점검용으로 확인

In [1]:
# konlpy : 한국어 형태소 분석 라이브러리 (Java 기반)
!pip install konlpy


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# GPU 인식 확인 (환경 점검용)
import torch

print(torch.cuda.is_available())        # True 나오면 GPU 인식됨
print(torch.cuda.get_device_name(0))    # GPU 이름
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

True
NVIDIA GeForce RTX 5060 Ti
cuda


### 텍스트 마이닝 5과정

| 단계 | 내용 | 오늘 사용한 도구 |
|---|---|---|
| 1. 데이터 수집 | crawling, 데이터 포털에서 말뭉치 확보 | UnSmile 데이터셋 (tsv) |
| 2. 데이터 정제 (전처리) | 불필요한 데이터를 제거<br>ex) dog, 강아지, 개 → 강아지 | `re`, `kiwipiepy`, `emoji` |
| 3. 토큰화 | 텍스트를 형태소 단위로 분리<br>품사를 태깅해서 필요한 품사만 추출 | `konlpy` (Okt) |
| 4. 특징 값 추출 | 중요한 토큰과 불필요한 토큰을 판단 | `TfidfVectorizer` |
| 5. 데이터 분석 (모델링) | 데이터를 학습시켜 새로운 데이터를 잘 예측하게 만드는 것이 목적 | `LogisticRegression` |

---
## 1. 데이터 수집 (불러오기)

- UnSmile 데이터셋은 **멀티라벨** 구조 : 문장 1개 + 혐오 카테고리 10개 컬럼
- 오늘은 그중 `clean` 컬럼만 사용해서 **혐오 / 정상 이진 분류**로 단순화

In [3]:
import pandas as pd

# 구분자에 따라 파일 형식이 다름
# csv : comma(,)로 구분된 파일
# tsv : tab(\t)으로 구분된 파일 -> sep="\t" 를 꼭 지정해야 한다

train = pd.read_csv("data/unsmile_train_v1.0.tsv", sep="\t")   # 학습용 15,005건
test = pd.read_csv("data/unsmile_valid_v1.0.tsv", sep="\t")    # 평가용 3,737건

In [4]:
# 데이터 확인
# 문장 1개 + 혐오 카테고리 10개 컬럼으로 구성된 멀티라벨 구조
# clean == 1 -> 혐오 표현이 없는 정상 문장
train.head(10)

,문장,여성/가족,남성,성소수자,인종/국적,연령,지역,종교,기타 혐오,악플/욕설,clean,개인지칭
0,일안하는 시간은 쉬고싶어서 그런게 아닐까,0,0,0,0,0,0,0,0,0,1,0
1,아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...,0,0,0,0,0,0,1,0,0,0,0
2,루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o doin 진짜 띵...,0,0,0,0,0,0,0,0,0,1,0
3,홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...,0,0,0,0,0,0,0,0,0,1,0
4,아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...,1,0,0,0,0,0,0,0,0,0,0
5,고향가서 피방가면 동네 부럴 친구들이랑은 뭐 거르는 거 없이 이야기하니까 막 말하게...,0,0,0,0,0,0,0,0,0,1,0
6,예수 십새끼 개새끼 창녀아들 애비실종 가정교육 못받은 무뇌충 허언증 정신병자 사기꾼...,0,0,0,1,0,1,1,1,0,0,0
7,나이쳐먹고 피시방가는 놈들은 대가리에 똥만찬 놈들임,0,0,0,0,0,0,0,0,1,0,0
8,출산청같은 소리하네. 우리나라 계집들 다 의무적으로 군대보내면 해결될 일이다. 정신...,1,0,0,0,0,0,0,0,0,0,0
9,재능 ㅂㅅㅌㅊ노 ㅋㅋㅋㅋㅋ나도 합성배우고싶다이기,0,0,0,0,0,0,0,0,0,1,0


In [5]:
# 평가 데이터도 같은 컬럼 구조인지 확인
test.head()

,문장,여성/가족,남성,성소수자,인종/국적,연령,지역,종교,기타 혐오,악플/욕설,clean,개인지칭
0,ㅇㄱㄹㅇ 진짜 죽어도 상관없다는 마인드로 싸웠더니 지금 서열 상타취노 식칼들고 니가...,0,1,0,0,0,0,0,0,0,0,0
1,여자들은 취미가 애낳는건가.. 취미를 좀 가져라,1,0,0,0,0,0,0,0,0,0,0
2,개슬람녀 다 필요없고 니 엄마만 있으면 된다,0,0,0,1,0,0,1,0,0,0,0
3,조팔ㅋㅋ 남한 길거리 돌아다니면 한국남자때문에 눈재기하는데 그걸 내 폰에 굳이 담아...,0,1,0,0,0,0,0,0,0,0,0
4,바지 내리다 한남들 와꾸 보고 올려뿟노,0,1,0,0,0,0,0,0,0,0,0


In [6]:
# 특정 카테고리만 필터링해서 확인 (예: 지역 혐오 문장)
train[train['지역'] == 1]

,문장,여성/가족,남성,성소수자,인종/국적,연령,지역,종교,기타 혐오,악플/욕설,clean,개인지칭
6,예수 십새끼 개새끼 창녀아들 애비실종 가정교육 못받은 무뇌충 허언증 정신병자 사기꾼...,0,0,0,1,0,1,1,1,0,0,0
25,홍어무침 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,0,0,0,0,0,1,0,0,0,0,0
28,홍들 피들 폭들 ㅎㅎㅎ,0,0,0,0,0,1,0,0,0,0,0
35,이새끼 전라도,0,0,0,0,0,1,0,0,0,0,0
37,진짜 전라도 존나 싫다,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
14956,경상도새끼네,0,0,0,0,0,1,0,0,0,0,0
14975,7시말은 왜이렇게 듣기가 싫냐. 식당 같은데 가서도 전라도 말 들리면 신경 엄청 곤...,0,0,0,0,0,1,0,0,0,0,0
14979,왜 재밌겠는데 뭘 섬라도 쪽 꿀잼뉴스 공중제비하고 보겠네,0,0,0,0,0,1,0,0,0,0,0
14998,전라디언은 뇌구조가 다르게 태어난당께,0,0,0,0,0,1,0,0,0,0,0


---
## 2. 데이터 정제 (전처리)

댓글 데이터라 노이즈가 많음 → 3단계로 정제한다.

| 순서 | 처리 | 도구 |
|---|---|---|
| 1 | 특수문자 · 자모(ㅋㅋ, ㅠㅠ) 제거 | `re` (정규표현식) |
| 2 | 띄어쓰기 교정 | `kiwipiepy` |
| 3 | 이모지 제거 | `emoji` |

> **포인트** : train / test에 **완전히 동일한 정제**를 적용해야 하므로 마지막에 함수(`text_cleaning`)로 묶는다.

In [9]:
# 문장 데이터 추출하기
# 분석에 사용하는 것은 '문장' 컬럼 하나뿐

train['문장']

# 데이터를 받아줄 리스트
# -> 이후 전처리 함수들이 리스트 단위로 동작하기 때문에 리스트로 담아둔다
train_doc = []

for doc in train['문장']:
    train_doc.append(doc)

test_doc = []

for doc in test['문장']:
    test_doc.append(doc)

# 참고 : train['문장'].tolist() 한 줄로도 같은 결과

In [16]:
# [정제 1] 정규표현식 연습
# 불필요한 데이터 삭제
# -> 의미가 없는 특수문자, 완성되지 않은 글자(자모)
# 특수문자와 완성되지 않은 글자를 찾아내는 패턴을 작성
# 정규표현식을 활용해서 문자 패턴을 작성하자

import re

# re.compile : 찾고 싶은 문자 패턴을 미리 만들어두는 함수
# ㅋ을 찾아내는 패턴 작성
pattern = re.compile('[ㅋ]')
pattern

# 예시 데이터
sample = '오늘의 점심 메뉴는 라면이다 ㅋㅋㅋㅋㅋ'

# sample 문장에서 ㅋ을 ㅎ으로 바꾸기
# re.sub(패턴, 바꿀 문자, 문장) -> 패턴으로 찾은 부분을 다른 텍스트로 치환
re.sub(pattern, 'ㅎ', sample)

'오늘의 점심 메뉴는 라면이다 ㅎㅎㅎㅎㅎ'

In [21]:
# [정제 1 적용] 불필요한 문자를 실제 데이터에서 제거

# 1. 패턴 작성
# [^...] : 대괄호 안의 문자를 '제외한' 나머지를 찾는다는 의미
# 남길 문자 -> 영문(a-zA-Z), 숫자(0-9), 완성형 한글(가-힣), 공백(\s), 문장부호(. ? !)
# 즉, 그 외의 특수문자 / 자모(ㅋ, ㅠ) / 이모지는 모두 제거 대상
pattern = re.compile(r'[^a-zA-Z0-9가-힣\s\.\?\!]')

# 정규표현식으로 전처리한 데이터를 저장
train_re = []

# 2. 문장 하나하나에 대해 패턴을 적용하여 데이터 삭제
for doc in train_doc:
    result = re.sub(pattern, '', doc)
    train_re.append(result)

# test 데이터도 적용 (train과 반드시 같은 기준으로 처리)

test_re = []
for doc in test_doc:
    result = re.sub(pattern, '', doc)
    test_re.append(result)

In [23]:
# kiwipiepy : 띄어쓰기 교정 등 한국어 전처리를 도와주는 라이브러리
!pip install kiwipiepy


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
# [정제 2] 띄어쓰기 교정
# 불필요한 용어를 1차 제거하면 단어가 붙어버리는 경우가 생김
# -> 띄어쓰기를 교정해서 형태소 분석 정확도를 높인다

# 띄어쓰기를 해주는 인공지능 활용
# -> kiwipiepy

from kiwipiepy import Kiwi

kiwi = Kiwi()

# space() : 문장의 띄어쓰기를 교정하는 기능
kiwi.space('아버지가방에들어가신다')

'아버지가 방에 들어가신다'

In [27]:
# emoji : 이모지를 찾아내고 치환해주는 라이브러리
!pip install emoji

  Obtaining dependency information for emoji from https://files.pythonhosted.org/packages/e1/5e/4b5aaaabddfacfe36ba7768817bd1f71a7a810a43705e531f3ae4c690767/emoji-2.15.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   -- ------------------------------------- 30.7/608.4 kB 1.3 MB/s eta 0:00:01
   -------------------- ------------------- 307.2/608.4 kB 4.8 MB/s eta 0:00:01
   ---------------------------------------  604.2/608.4 kB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 608.4/608.4 kB 4.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# [정제 3] 이모티콘 제거
# 정규표현식으로도 대부분 걸러지지만, 이모지는 전용 라이브러리가 더 안전하다

import emoji

sample = "😀😁😂🤣😃😋😎🥲🥰😘🤨☺️😶🫥 이모지"

# replace_emoji : 이모지를 찾아 다른 문자(기본값 '')로 치환
emoji.replace_emoji(sample)

' 이모지'

In [37]:
# 텍스트 데이터 정제
# 1. 정규표현식으로 불필요한 텍스트를 제거
# 2. kiwipiepy를 활용한 띄어쓰기 교정
# 3. emoji를 활용한 이모지 삭제

# -> 위의 3가지를 한 번에 진행하는 사용자 함수 작성
# -> 함수로 묶어두면 train / test에 똑같은 처리를 보장할 수 있다

def text_cleaning(document):
    # 1. 정규표현식으로 불필요한 텍스트를 제거
    pattern = re.compile(r'[^a-zA-Z0-9가-힣\s\.\?\!]')
    result = []

    for doc in document:
        re_doc = re.sub(pattern, '', doc)
        # 2. kiwipiepy를 활용한 띄어쓰기 교정
        space_doc = kiwi.space(re_doc)
        # 3. emoji를 활용한 이모지 삭제
        emoji_doc = emoji.replace_emoji(space_doc)
        # 4. result라는 리스트에 저장
        result.append(emoji_doc)

    return result

In [39]:
# 함수가 잘 작동하는지 확인 (데이터가 많아 시간이 조금 걸림)
train_clean = text_cleaning(train_doc)
test_clean = text_cleaning(test_doc)

---
## 3. 토큰화 (형태소 분석)

### 형태소 분석기 비교

- **한국어** : konlpy (Okt, Kkma, Komoran, Hannanum, Mecab)
- **영어** : NLTK 등
- 공식 문서 : https://konlpy.org/ko/latest/index.html
- Mecab (Colab 설치용) : https://github.com/SOMJANG/Mecab-ko-for-Google-Colab

> 이번 실습은 댓글/SNS 문체에 강하고 속도가 빠른 **Okt**를 사용

| 분석기    | 속도        | 정확도        | 특징                                                         | 장단점                                                         |
|-----------|-------------|---------------|--------------------------------------------------------------|----------------------------------------------------------------|
| **Mecab** | 매우 빠름   | 높음          | - 백터 기반의 빠른 분석<br>- 대용량 데이터 처리에 적합          | + **장점:** 속도 및 효율성 우수<br>- **단점:** 설치가 다소 복잡, 사전 관리 필요 |
| **Kkma**  | 느림        | 높음          | - 문장 단위의 세밀한 분석 제공<br>- 다양한 형태소 정보 리턴       | + **장점:** 상세 분석, 문맥 파악 용이<br>- **단점:** 속도 느림, 대용량 데이터에 부적합 |
| **Hannanum** | 보통     | 중간          | - KAIST 개발 분석기로 전통적 접근법 사용<br>- 기본 문법 규칙 기반  | + **장점:** 비교적 안정적인 결과 제공<br>- **단점:** 문맥 반영 미흡, 업데이트 한계  |
| **Komoran** | 보통       | 높음          | - 최신 알고리즘 일부 적용<br>- 딥러닝 요소 도입 가능             | + **장점:** 높은 정확도, 견고한 성능<br>- **단점:** 옵션 및 커스터마이징 제한       |
| **Okt**   | 빠름        | 중간 ~ 높음   | - 소셜 미디어(예: 트위터) 문체에 최적화<br>- 감성 분석 등 특화       | + **장점:** 사용법 간편, 빠른 처리<br>- **단점:** 복잡한 문장 분석에는 한계, 단어 세분화 미흡 |

---

| 형태소 분석기                | 주요 메서드 및 인자                                            | 인자 설명                                                                                                 |
|-----------------------------|--------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------|
| **Hannanum**                | - `pos(phrase, ntags=9, flatten=True)`                         | - **ntags**: 태그의 상세도를 지정 (일반적으로 9 또는 22 사용) <br> - **flatten**: 결과를 단일 리스트로 반환 여부  |
|                             | - `analyze(phrase)`                                             | - 후보 분석 결과(여러 형태소 분석 후보)를 반환                                                              |
| **Kkma**                    | - `pos(phrase, flatten=True)`                                  | - **flatten**: 결과를 평탄화하여 하나의 리스트로 반환                                                         |
|                             | - `sentences(phrase)`                                          | - 입력 텍스트를 문장 단위로 분리하여 리스트로 반환                                                            |
|                             | - `nouns(phrase)`, `morphs(phrase)`                              | - 각각 명사와 모든 형태소만을 추출                                                                           |
| **Komoran**                 | - `pos(phrase, flatten=True)`                                  | - **flatten**: 결과를 단일 리스트로 반환 여부                                                                 |
|                             | - `nouns(phrase)`, `morphs(phrase)`                              | - 각각 명사와 모든 형태소만을 추출                                                                           |
| **Mecab**                   | - `pos(phrase, flatten=True)`                                  | - **flatten**: 결과를 평탄화하여 하나의 리스트로 반환                                                         |
|                             | - `nouns(phrase)`, `morphs(phrase)`                              | - 각각 명사와 모든 형태소만을 추출                                                                           |
| **Okt (Open Korean Text)**  | - `pos(phrase, norm=False, stem=False)`                        | - **norm**: 정규화 여부 (예: 숫자, 영문 등의 표준 형태로 변환할지 결정) <br> - **stem**: 어간 추출 여부 (어근화)  |
|                             | - `phrases(phrase)`                                            | - 텍스트 내 구(phrase)를 추출                                                                               |
|                             | - `nouns(phrase)`, `morphs(phrase)`                              | - 각각 명사와 모든 형태소만을 추출                                                                           |

In [41]:
# konlpy 재확인 (이미 설치되어 있으면 넘어간다)
!pip install konlpy


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# 토큰화
# -> 말뭉치를 형태소 단위로 잘게 쪼개주는 작업
# -> 필요한 형태소만 가져다 쓸 수 있게하기 위함

# konlpy (Korean Language Processing in Python)
# -> 한국어 텍스트 데이터를 형태소로 분석해주는 도구

import konlpy

In [45]:
# konlpy는 Java 기반이라 JDK 경로(JAVA_HOME)를 잡아줘야 동작한다
# -> 본인 PC에 설치된 JDK 경로로 수정 필요
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-25.0.4.7-hotspot"

In [46]:
# konlpy 안에는 여러가지 한국어 데이터 분석기가 있다

# 형태소 분석기 생성
from konlpy.tag import Okt, Kkma

okt = Okt()     # 빠르고 SNS 문체에 강함 -> 이번 실습에 사용
kkma = Kkma()   # 느리지만 세밀하게 분석

In [49]:
# morphs -> 문장을 형태소 단위로만 쪼갬

okt.morphs("아버지가 방에 들어가신다.")
kkma.morphs("아버지가 방에 들어가신다.")

# pos -> 문장을 형태소 단위로 쪼개면서 품사까지 태깅
okt.pos("아버지가 방에 들어가신다.")
kkma.pos("아버지가 방에 들어가신다.")

[('아버지', 'NNG'),
 ('가', 'JKS'),
 ('방', 'NNG'),
 ('에', 'JKM'),
 ('들어가', 'VV'),
 ('시', 'EPH'),
 ('ㄴ다', 'EFN'),
 ('.', 'SF')]

In [50]:
# 각 형태소마다 매겨지는 품사의 종류(태그 체계)가 분석기마다 다름
# Okt  -> Noun / Verb / Adjective ...
# Kkma -> NNG / VV / VA ...

okt.tagset

kkma.tagset

{'EC': '연결 어미',
 'ECD': '의존적 연결 어미',
 'ECE': '대등 연결 어미',
 'ECS': '보조적 연결 어미',
 'EF': '종결 어미',
 'EFA': '청유형 종결 어미',
 'EFI': '감탄형 종결 어미',
 'EFN': '평서형 종결 어미',
 'EFO': '명령형 종결 어미',
 'EFQ': '의문형 종결 어미',
 'EFR': '존칭형 종결 어미',
 'EP': '선어말 어미',
 'EPH': '존칭 선어말 어미',
 'EPP': '공손 선어말 어미',
 'EPT': '시제 선어말 어미',
 'ET': '전성 어미',
 'ETD': '관형형 전성 어미',
 'ETN': '명사형 전성 어미',
 'IC': '감탄사',
 'JC': '접속 조사',
 'JK': '조사',
 'JKC': '보격 조사',
 'JKG': '관형격 조사',
 'JKI': '호격 조사',
 'JKM': '부사격 조사',
 'JKO': '목적격 조사',
 'JKQ': '인용격 조사',
 'JKS': '주격 조사',
 'JX': '보조사',
 'MA': '부사',
 'MAC': '접속 부사',
 'MAG': '일반 부사',
 'MD': '관형사',
 'MDN': '수 관형사',
 'MDT': '일반 관형사',
 'NN': '명사',
 'NNB': '일반 의존 명사',
 'NNG': '보통명사',
 'NNM': '단위 의존 명사',
 'NNP': '고유명사',
 'NP': '대명사',
 'NR': '수사',
 'OH': '한자',
 'OL': '외국어',
 'ON': '숫자',
 'SE': '줄임표',
 'SF': '마침표, 물음표, 느낌표',
 'SO': '붙임표(물결,숨김,빠짐)',
 'SP': '쉼표,가운뎃점,콜론,빗금',
 'SS': '따옴표,괄호표,줄표',
 'SW': '기타기호 (논리수학기호,화폐기호)',
 'UN': '명사추정범주',
 'VA': '형용사',
 'VC': '지정사',
 'VCN': "부정 지정사, 형용사 '아니다'",
 'VC

In [53]:
# 하나의 문장이 쪼개져서 리스트로 변환됨
# -> 리스트로 넣으면 각자 다른 데이터로 인식하기 때문에 하나의 문자열로 합쳐주기
#    (TF-IDF는 '문장 하나 = 문자열 하나' 형태를 입력으로 받는다)
['아버지' ,'방','들어가신다']
" ".join(['아버지' ,'방','들어가신다'])

'아버지 방 들어가신다'

In [55]:
# [train 토큰화]
# 토큰화가 진행된 형태소를 담을 리스트
train_morphs = []

# 1. 말뭉치 데이터에 대해 형태소 분석
for doc in train_clean :

    # 2. pos 함수로 형태소 분석 + 품사 태깅
    # 품사가 Noun, Verb, Adjective인 데이터만 저장
    morphs = okt.pos(doc)

    # 문장 안에서 필요한 형태소만 저장
    morphs_list = []
    for text in morphs :
        # 형태소가 하나하나씩 반복돼서 text에 담김
        # text[0] : 형태소 텍스트
        # text[1] : 품사
        # 조사 / 어미는 의미가 없으므로 명사, 형용사, 동사만 사용
        if text[1] in ['Noun', 'Adjective', 'Verb'] :
        # text[0]를 morphs_list에 저장!
            morphs_list.append(text[0])
    # 리스트로 묶여져 있는 각각의 형태소를 한 문자열로 합치기
    result = " ".join(morphs_list)
    # 문자열로 바뀐 데이터를 저장
    train_morphs.append(result)

In [ ]:
# [test 토큰화] train과 완전히 동일한 기준으로 처리
# 토큰화가 진행된 형태소를 담을 리스트
test_morphs = []

# 1. 말뭉치 데이터에 대해 형태소 분석
for doc in test_clean :
    # 2. pos 함수로 형태소 분석 + 품사 태깅
    # 품사가 Noun, Verb, Adjective인 데이터만 저장
    morphs = okt.pos(doc)

    # 문장 안에서 필요한 형태소만 저장
    morphs_list = []
    for text in morphs :
        # text[0] : 형태소 텍스트 / text[1] : 품사
        if text[1] in ['Noun', 'Adjective', 'Verb'] :
            morphs_list.append(text[0])
    # 리스트로 묶여져 있는 각각의 형태소를 한 문자열로 합치기
    result = " ".join(morphs_list)
    # 문자열로 바뀐 데이터를 저장
    test_morphs.append(result)

In [ ]:
# 토큰화 결과 확인 (조사 / 어미가 사라지고 핵심 단어만 남음)
# 전체를 출력하면 15,005건이 다 찍히므로 앞 10개만 확인
train_morphs[:10]

---
## 4. 특징값 추출 (TF-IDF)

- 컴퓨터는 문자열을 못 읽으므로 **숫자 벡터**로 바꿔야 한다
- TF-IDF는 그 과정에서 **단어의 중요도**까지 함께 반영한다
  - 모든 문서에 흔한 단어 → 중요도 ↓
  - 특정 문서에만 자주 나오는 단어 → 중요도 ↑
- `fit`은 **train에만**, `transform`은 train/test 모두 (test 정보 유출 방지)

In [60]:
# TF-IDF : 단어(형태소)의 중요성을 계산하기 위해 사용하는 수식
# TF  (Term Frequency)           : 한 문서에서 그 단어가 얼마나 자주 나오는가
# IDF (Inverse Document Frequency) : 전체 문서에서 얼마나 희귀한가

# 전체적으로 자주 등장하는 단어 : 중요하지 않다
# 특정 문서에서만 자주 등장하는 단어 : 중요함

sample = ['안녕하세요 파이썬 수업',
          '안녕하세요 html 수업',
          '안녕하세요 머신러닝 수업',
          '안녕하세요 반갑습니다']

# TF-IDF 계산 도구를 호출
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

# 문장 데이터를 학습 (단어 사전 + 각 단어의 IDF 값을 만든다)
print(tfidf.fit(sample))

TfidfVectorizer()


In [62]:
# transform : 학습한 기준대로 문장을 숫자 벡터로 변환
sample_trans = tfidf.transform(sample)

sample_trans.toarray()

# DataFrame으로 정리
# -> 모든 문장에 있는 '안녕하세요'는 값이 작고,
#    한 문장에만 있는 '파이썬', '반갑습니다'는 값이 큰 것을 확인
pd.DataFrame(sample_trans.toarray(),
             columns = tfidf.get_feature_names_out()
             )

,html,머신러닝,반갑습니다,수업,안녕하세요,파이썬
0,0.000000,0.000000,0.000000,0.492489,0.402642,0.771579
1,0.771579,0.000000,0.000000,0.492489,0.402642,0.000000
2,0.000000,0.771579,0.000000,0.492489,0.402642,0.000000
3,0.000000,0.000000,0.886548,0.000000,0.462637,0.000000


In [ ]:
# train을 활용해서 tfidf 모델을 학습

tfidf = TfidfVectorizer()

# train 데이터 안에 있는 문장을 토대로 단어의 중요성을 학습
# fit은 반드시 train에만! (test까지 fit하면 평가 데이터 정보가 새어 들어간다)
print(tfidf.fit(train_morphs))

TfidfVectorizer()


In [65]:
# 학습 데이터, 평가 데이터 만들기
# transform만 적용 -> train에서 만든 단어 사전 기준을 그대로 사용

X_train = tfidf.transform(train_morphs).toarray()
X_test = tfidf.transform(test_morphs).toarray()

---
## 5. 모델링 & 평가

- 정답(label) : `clean` 컬럼 → `0 = 혐오`, `1 = 정상`
- 모델 : **로지스틱 회귀** (텍스트 분류 베이스라인으로 빠르고 해석이 쉬움)

In [76]:
# 정답(label) 만들기
# clean에 1값이 있으면 혐오 문장 X (정상)
# clean에 0값이 있으면 혐오 문장 O
# -> 10개 카테고리 중 clean만 사용해서 이진 분류로 단순화

dict1 ={0 : '혐오', 1 : '정상'}

y_train = train['clean'].map(dict1)
y_test = test['clean'].map(dict1)

In [77]:
# LR(Logistic Regression)
# 텍스트 분류의 기본 베이스라인 모델 (빠르고 해석이 쉽다)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
print(model.fit(X_train, y_train))

LogisticRegression()


In [78]:
# 내가 쓴 문장이 혐오인지 혐오가 아닌지 예측
text = ["이게 왜 욕이노ㅋㅋ"]

# 우리가 예측하고자 하는 데이터도
# 1. 데이터 정제
# 2. 형태소 분석
# 3. tf-idf로 수치화
# ※ 아래 코드는 3번만 적용된 상태 (1, 2번이 빠져 있음) -> 다음 셀에서 보완
sample = tfidf.transform(text).toarray()

model.predict(sample)

array(['혐오'], dtype=object)

In [ ]:
# [개선] 학습과 똑같은 전처리 순서를 거쳐서 예측하는 함수
# 정제 / 토큰화를 건너뛰면 학습한 단어 사전과 형태가 달라져 예측이 부정확해진다

def predict_hate(text_list):
    # 1. 데이터 정제
    cleaned = text_cleaning(text_list)

    # 2. 형태소 분석 + 품사 필터링 (train과 동일 기준)
    morphs_result = []
    for doc in cleaned:
        morphs_list = [t[0] for t in okt.pos(doc)
                       if t[1] in ['Noun', 'Adjective', 'Verb']]
        morphs_result.append(" ".join(morphs_list))

    # 3. tf-idf로 수치화 (fit 없이 transform만!)
    X = tfidf.transform(morphs_result).toarray()

    # 4. 예측
    return model.predict(X)


predict_hate(["이게 왜 욕이노ㅋㅋ", "오늘 점심 맛있었다"])

In [79]:
# 평가 : 정확도(accuracy) 확인
# 약 0.77 -> 베이스라인 수준
# 단, 혐오 75% / 정상 25%로 불균형한 데이터이므로
# classification_report로 precision, recall도 같이 봐야 한다
model.score(X_test, y_test)

0.7698688787797698

---
## 6. 오늘의 결과 정리

### 결과
- test 정확도(accuracy) **약 0.770**
- 데이터가 혐오 75% / 정상 25%로 불균형 → 정확도만 보면 과대평가될 수 있음
  (`classification_report`로 precision / recall / f1 확인 필요)

### 배운 것
1. 텍스트마이닝 5단계(수집 → 정제 → 토큰화 → 특징값 추출 → 모델링) 흐름을 한 번에 실습
2. 정규표현식 `[^...]` 로 "남길 문자만 지정"하는 방식이 훨씬 편하다
3. `konlpy`는 Java 기반이라 `JAVA_HOME` 설정이 선행되어야 한다
4. 조사·어미를 버리고 **명사 / 동사 / 형용사**만 남기면 노이즈가 크게 줄어든다
5. `fit`은 train에만 적용해야 한다 (test까지 fit하면 데이터 유출)

### 다음에 시도해볼 것
- `classification_report`, confusion matrix로 지표 보강
- `TfidfVectorizer(ngram_range=(1,2), min_df=2)` 등 파라미터 튜닝
- 불용어(stopwords) 사전 추가
- 다른 모델 비교 (LinearSVC, ComplementNB) / KoBERT 같은 사전학습 모델
- `clean` 이진 분류 → 카테고리별 멀티라벨 분류로 확장